In [2]:
import pandas as pd 
import sqlite3

# Connexion à la base de données SQLite (elle sera créée si elle n'existe pas encore)
conn = sqlite3.connect("../yelp.db")

In [7]:
# Chargement du fichier business.json (petit, chargement direct possible)
df_business = pd.read_json("../data/yelp_academic_dataset_business.json", lines=True)

# Les colonnes 'attributes' et 'hours' contiennent des dictionnaires imbriqués
# SQLite ne sait pas stocker ça directement, on les convertit en texte (JSON) pour l'instant
df_business['attributes'] = df_business['attributes'].astype(str)
df_business['hours'] = df_business['hours'].astype(str)

df_business.shape

(150346, 14)

In [8]:
# Insertion dans la base de données SQLite (crée la table 'business')
df_business.to_sql("business", conn, if_exists="replace", index=False)

150346

In [9]:
# Chargement et insertion de tip.json
df_tip = pd.read_json("../data/yelp_academic_dataset_tip.json", lines=True)
df_tip.to_sql("tip", conn, if_exists="replace", index=False)
print("tip:", df_tip.shape)

# Chargement et insertion de checkin.json
df_checkin = pd.read_json("../data/yelp_academic_dataset_checkin.json", lines=True)
df_checkin.to_sql("checkin", conn, if_exists="replace", index=False)
print("checkin:", df_checkin.shape)

tip: (908915, 5)
checkin: (131930, 2)


In [11]:
# Chargement de user.json par morceaux (chunks) pour ne pas saturer la RAM
chunk_size = 100000
total_rows = 0

for i, chunk in enumerate(pd.read_json("../data/yelp_academic_dataset_user.json", lines=True, chunksize=chunk_size)):
    # 'friends' et 'elite' sont des listes, SQLite ne les stocke pas directement, on les convertit en texte
    chunk['friends'] = chunk['friends'].astype(str)
    chunk['elite'] = chunk['elite'].astype(str)

    # Premier chunk : on crée la table. Chunks suivants : on ajoute (append) à la suite
    if i == 0:
        chunk.to_sql("user", conn, if_exists="replace", index=False)
    else:
        chunk.to_sql("user", conn, if_exists="append", index=False)

    total_rows += len(chunk)
    print(f"Chunk {i+1} inséré, total lignes : {total_rows}")

Chunk 1 inséré, total lignes : 100000
Chunk 2 inséré, total lignes : 200000
Chunk 3 inséré, total lignes : 300000
Chunk 4 inséré, total lignes : 400000
Chunk 5 inséré, total lignes : 500000
Chunk 6 inséré, total lignes : 600000
Chunk 7 inséré, total lignes : 700000
Chunk 8 inséré, total lignes : 800000
Chunk 9 inséré, total lignes : 900000
Chunk 10 inséré, total lignes : 1000000
Chunk 11 inséré, total lignes : 1100000
Chunk 12 inséré, total lignes : 1200000
Chunk 13 inséré, total lignes : 1300000
Chunk 14 inséré, total lignes : 1400000
Chunk 15 inséré, total lignes : 1500000
Chunk 16 inséré, total lignes : 1600000
Chunk 17 inséré, total lignes : 1700000
Chunk 18 inséré, total lignes : 1800000
Chunk 19 inséré, total lignes : 1900000
Chunk 20 inséré, total lignes : 1987897


In [12]:
# Chargement de review.json par morceaux (le plus gros fichier)
chunk_size = 100000
total_rows = 0

for i, chunk in enumerate(pd.read_json("../data/yelp_academic_dataset_review.json", lines=True, chunksize=chunk_size)):
    if i == 0:
        chunk.to_sql("review", conn, if_exists="replace", index=False)
    else:
        chunk.to_sql("review", conn, if_exists="append", index=False)

    total_rows += len(chunk)
    print(f"Chunk {i+1} inséré, total lignes : {total_rows}")

Chunk 1 inséré, total lignes : 100000
Chunk 2 inséré, total lignes : 200000
Chunk 3 inséré, total lignes : 300000
Chunk 4 inséré, total lignes : 400000
Chunk 5 inséré, total lignes : 500000
Chunk 6 inséré, total lignes : 600000
Chunk 7 inséré, total lignes : 700000
Chunk 8 inséré, total lignes : 800000
Chunk 9 inséré, total lignes : 900000
Chunk 10 inséré, total lignes : 1000000
Chunk 11 inséré, total lignes : 1100000
Chunk 12 inséré, total lignes : 1200000
Chunk 13 inséré, total lignes : 1300000
Chunk 14 inséré, total lignes : 1400000
Chunk 15 inséré, total lignes : 1500000
Chunk 16 inséré, total lignes : 1600000
Chunk 17 inséré, total lignes : 1700000
Chunk 18 inséré, total lignes : 1800000
Chunk 19 inséré, total lignes : 1900000
Chunk 20 inséré, total lignes : 2000000
Chunk 21 inséré, total lignes : 2100000
Chunk 22 inséré, total lignes : 2200000
Chunk 23 inséré, total lignes : 2300000
Chunk 24 inséré, total lignes : 2400000
Chunk 25 inséré, total lignes : 2500000
Chunk 26 inséré, t

In [16]:
df_checkin = pd.read_json("../data/yelp_academic_dataset_checkin.json", lines=True)
df_checkin.to_sql("checkin", conn, if_exists="replace", index=False)
print("checkin:", df_checkin.shape)

checkin: (131930, 2)


In [17]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

       name
0  business
1       tip
2      user
3    review
4   checkin
